In [ ]:
import pandas as pd
import plotly
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import pickle
import sys
import os
# Add the parent directory (Codes) to sys.path
sys.path.append(os.path.abspath('../'))

from utils.datautils import Readdataset, calculate_dataset_metrics, Splitview
from utils.train_utils import Evaluate_model

In [ ]:
print("Current working directory:", os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
from utils.modelutils import *

In [ ]:
model_path = '../../Tree_Models/SanityCheck_learned_tree.pkl'
model = pickle.load(open(model_path, 'rb'))

In [ ]:
Xtrain, ytrain, Xval, yval, Xtest, ytest, N, T = load_preprocessed_data('SanityCheck')
print_labels(ytest)

#code for loading simulated data
# # Load simulated peak valley data
# with open("../peak_valley_simulated_data.pkl", "rb") as f:
#     Xtrain, ytrain, Xval, yval, Xtest, ytest = pickle.load(f)

In [ ]:
get_node_properties(model, 0)

In [ ]:
print_tree_structure(model)

In [ ]:
print_tree_summary(model)

In [ ]:
print_all_leaf_rules(model)

In [ ]:
# key_params = extract_key_features(model[0].bestmodel)
key_params = extract_key_features(model[0].bestmodel, view_names=["Original"])

In [ ]:
# key_params['agg']

In [ ]:
print(key_params['Original']['u'].shape)

In [ ]:
print(key_params['Original']['top_indices'])
print(key_params['Original']['A'][:,key_params['Original']['top_indices']])
print(key_params['Original']['u'])

Manually Calculate Robustness Value

In [ ]:
model

In [ ]:
import torch.nn.functional as F
import torch
from Models_node_copy  import clamp
test_data = []
for i in ytest:
    if i == 1:
        test_data.append(np.full(shape = (1,20), fill_value=30))
    else:
        test_data.append(np.full(shape = (1,20), fill_value=50))
test_data = torch.tensor(test_data).unsqueeze(0)
r_a1 = test_data * model[0].bestmodel.t1 - model[0].bestmodel.b1
r_asgm1 = torch.sigmoid(r_a1)
A_sm1 = F.softmax(model[0].bestmodel.A1, dim=1)
weightbias1 = 1-model[0].bestmodel.beta1 + torch.sum(A_sm1 * (r_asgm1), 1)
activate1 = clamp(weightbias1).reshape([-1,1])

In [ ]:
test_data

In [ ]:
A_sm1

In [ ]:
weightbias1

In [ ]:
activate1

In [ ]:
model[0].bestmodel(test_data)

In [ ]:
print(key_params['FFT']['top_indices'])
print(key_params['FFT']['A'][:,key_params['FFT']['top_indices']])
print(key_params['FFT']['u'])

In [ ]:
print(key_params['Derivative']['top_indices'])
print(key_params['Derivative']['A'][:,key_params['Derivative']['top_indices']])
print(key_params['Derivative']['u'])

In [ ]:
#
fig = go.Figure()

# Use red for label 1 and blue for label 0 (update colors as desired)
for i in range(Xtest.shape[0]):
    label = int(ytest[i])
    color = 'red' if label == 1 else 'blue'
    fig.add_trace(go.Scatter(
        x=np.arange(Xtest.shape[1]),
        y=Xtest[i],
        mode='lines',
        name=f'Waveform {i} (label {label})',
        line=dict(color=color)
    ))
    

fig.update_layout(
    title="Waveforms in Xtest with Corresponding ytest Labels",
    xaxis_title="Time Index",
    yaxis_title="Amplitude",
    template="plotly_white"
)
fig.show()

In [ ]:
print_labels(ytest)

In [ ]:
T = 20
Xori = Xtest[:, :T]
# Xpv = Xtest[:, T:T+10]
# Xder = Xtest[:, T+10:]

In [ ]:
# Xori, Xfft, Xder = Splitview(Xtest, T)

In [ ]:
#
fig = go.Figure()

# Use red for label 1 and blue for label 0 (update colors as desired)
for i in range(Xori.shape[0]):
    label = int(ytest[i])
    color = 'blue' if label == 1 else 'red'
    fig.add_trace(go.Scatter(
        x=np.arange(Xori.shape[1]),
        y=Xori[i],
        mode='lines',
        name=f'Waveform {i} (label {label})',
        line=dict(color=color)
    ))
    fig.add_hline(y=1.9626263e+01, line=dict(color='black', dash='dash'))
    fig.add_vline(x=13, line=dict(color='black', dash='dot'))
    # fig.add_vline(x=372, line=dict(color='black', dash='dot'))

fig.update_layout(
    title="Waveforms in Xori with Corresponding ytest Labels -> Top 5 Indices between [205, 325], rule: Xraw always >= 1.0390085",
    xaxis_title="Time Index",
    yaxis_title="Amplitude",
    template="plotly_white"
)
fig.show()

In [ ]:
#
fig = go.Figure()

# Use red for label 1 and blue for label 0 (update colors as desired)
for i in range(Xpv.shape[0]):
    label = int(ytest[i])
    color = 'blue' if label == 1 else 'red'
    fig.add_trace(go.Scatter(
        x=np.arange(Xpv.shape[1]),
        # x = np.fft.fftfreq(Xfft.shape[1], d=1/Xfft.shape[1])[:306],
        y=Xpv[i],
        mode='lines',
        name=f'Waveform {i} (label {label})',
        line=dict(color=color)
    ))
    fig.add_hline(y=-0.09183013, line=dict(color='black', dash='dash'))
    fig.add_vline(x=0, line=dict(color='black', dash='dot'))
    fig.add_vline(x=7, line=dict(color='black', dash='dot'))

fig.update_layout(
    title="Waveforms in Xpv with Corresponding ytest Labels -> Top 5 Indices between [0, 7], rule: Xfft always >= -0.09183013",
    xaxis_title="Time Index",
    yaxis_title="Amplitude",
    template="plotly_white"
)
fig.show()

In [ ]:
#
fig = go.Figure()

# Use red for label 1 and blue for label 0 (update colors as desired)
for i in range(Xder.shape[0]):
    label = int(ytest[i])
    color = 'blue' if label == 1 else 'red'
    fig.add_trace(go.Scatter(
        x=np.arange(Xder.shape[1]),
        y=Xder[i],
        mode='lines',
        name=f'Waveform {i} (label {label})',
        line=dict(color=color)
    ))
    fig.add_hline(y=-0.47153744, line=dict(color='black', dash='dash'))
    fig.add_vline(x=256, line=dict(color='black', dash='dot'))
    fig.add_vline(x=384, line=dict(color='black', dash='dot'))

fig.update_layout(
    title="Waveforms in Xder with Corresponding ytest Labels -> Top 5 Indices between [256, 384], rule: Xder always >= -0.47153744",
    xaxis_title="Time Index",
    yaxis_title="Amplitude",
    template="plotly_white"
)
fig.show()

Barcharts of weight values for different views

In [ ]:
# Squeeze the array in case it's 2D and get the 1D array of values
A_values = key_params['Original']['A'].squeeze()
indices = np.arange(len(A_values))

plt.figure(figsize=(10, 6))
plt.bar(indices, A_values, color='skyblue')
plt.xlabel('Index')
plt.ylabel('A Values')
plt.ylim(0, 0.01)  # Set y-axis from 0 to 1
plt.title("Contribution of Each Time Step in decision making - Raw data")
plt.show()

In [ ]:
# Squeeze the array in case it's 2D and get the 1D array of values
A_values = key_params['FFT']['A'].squeeze()
indices = np.arange(len(A_values))

plt.figure(figsize=(10, 6))
plt.bar(indices, A_values, color='skyblue')
plt.xlabel('Index')
plt.ylabel('A Values')
plt.ylim(0, 0.5)  # Set y-axis from 0 to 1
plt.title("Contribution of Each Time Step in decision making - Peak Valley data")
plt.show()

In [ ]:
# Squeeze the array in case it's 2D and get the 1D array of values
A_values = key_params['Derivative']['A'].squeeze()
indices = np.arange(len(A_values))

plt.figure(figsize=(10, 6))
plt.bar(indices, A_values, color='skyblue')
plt.xlabel('Index')
plt.ylabel('A Values')
plt.title("Contribution of Each Time Step in decision making - Derivative data")
plt.show()

In [ ]:
# Let's add a modified version of the Updateleftchd and Updaterigtchd functions 
# to understand why Node 2 still split despite having ginis=0

def examine_split_decision(node_id):
    node = model[node_id]
    print(f"Examining Node {node_id}:")
    print(f"  ginis: {getattr(node, 'ginis', 'Not set')}")
    print(f"  ginist: {getattr(node, 'ginist', 'Not set')}")
    print(f"  stoptrain: {getattr(node, 'stoptrain', 'Not set')}")
    
    # In train_utils.py, the condition for setting stoptrain=True is:
    # if ylginit == 0 or ylgini == 0:
    #     Nodes[maxnum].stoptrain = True
    
    print("\nAccording to the code logic:")
    if hasattr(node, 'ginis') and hasattr(node, 'ginist'):
        should_stop = (node.ginis == 0 or node.ginist == 0)
        print(f"  Should stop training based on gini values: {should_stop}")
        print(f"  Actual stoptrain value: {getattr(node, 'stoptrain', 'Not set')}")
        if should_stop != getattr(node, 'stoptrain', False):
            print("  ⚠️ INCONSISTENCY: stoptrain flag doesn't match expected value based on Gini indices")
    
    # Check if the node was split despite stoptrain=True
    has_children = hasattr(node, 'leftchd') or hasattr(node, 'rightchd')
    if getattr(node, 'stoptrain', False) and has_children:
        print("  ⚠️ INCONSISTENCY: Node was split despite stoptrain=True")
    elif not getattr(node, 'stoptrain', True) and not has_children:
        print("  ℹ️ Node was not split despite stoptrain=False")

# Examine Node 2
examine_split_decision(2)

# Also examine nodes 3 and 4 to see their properties
# print("\n" + "-"*50 + "\n")
# examine_split_decision(3)
# print("\n" + "-"*50 + "\n")
# examine_split_decision(4)

In [ ]:
# Explanation of why Node 2 was split despite ginis = 0

print("Explanation of the Node 2 inconsistency:")
print("\nIn the NSTSC algorithm, the stoptrain flag is set during node creation in Updateleftchd and Updaterigtchd.")
print("However, Node 2 was created before its Gini index was calculated and set to 0.")
print("\nSequence of events:")
print("  1. Node 0 is processed and split into Node 1 and Node 2")
print("  2. Node 1 is processed with no children (stoptrain=True)")
print("  3. Node 2 is processed and trained with bestmodel")
print("  4. After training, Node 2's ginis is set to 0")
print("  5. However, the node was already created with stoptrain=False")
print("  6. Build_tree continues to process Node 2, creating Nodes 3 and 4")
print("\nKey insight: The stoptrain flag was set BEFORE training the node,")
print("but the ginis value is set AFTER training. There's no code that updates")
print("stoptrain after the Gini index is determined from model training.")

print("\nLet's review the code flow:")
print("  1. Train model on Node 2 → Sets ginis to 0")
print("  2. No code updates stoptrain based on the trained model's Gini index")
print("  3. Tree building continues processing Node 2 since stoptrain is still False")

In [ ]:
# Simple tree visualization with detailed node information
def simple_tree_viz(tree):
    print("NSTSC Tree Structure:")
    print("===================")
    
    def print_node(node_id, indent=""):
        node = tree[node_id]
        print(f"{indent}Node {node_id}:")
        indent2 = indent + "  "
        
        # Print node attributes
        attrs = ["predcls", "bstmdlclass", "ginis", "ginist", "stoptrain"]
        for attr in attrs:
            if hasattr(node, attr):
                value = getattr(node, attr)
                if attr in ["ginis", "ginist"] and isinstance(value, (int, float)):
                    print(f"{indent2}{attr}: {value:.6f}")
                else:
                    print(f"{indent2}{attr}: {value}")
        
        # Print model info
        if hasattr(node, 'bestmodel'):
            from Models_node import TL_NN1, TL_NN2, TL_NN3, TL_NN4
            if isinstance(node.bestmodel, TL_NN1):
                model_name = "TL_NN1 (Conjunction/AND)"
            elif isinstance(node.bestmodel, TL_NN2):
                model_name = "TL_NN2 (Disjunction/OR)"
            elif isinstance(node.bestmodel, TL_NN3):
                model_name = "TL_NN3 (Always/Globally)"
            elif isinstance(node.bestmodel, TL_NN4):
                model_name = "TL_NN4 (Eventually/Finally)"
            else:
                model_name = "Unknown Model"
            print(f"{indent2}bestmodel: {model_name}")
    
    # Print all nodes with their connections
    for node_id in sorted(tree.keys()):
        print_node(node_id)
        print()
    
    # Print tree structure
    print("Tree Structure:")
    print("-" * 15)
    
    def print_children(node_id, indent=""):
        node = tree[node_id]
        print(f"{indent}Node {node_id}")
        next_indent = indent + "│   "
        last_indent = indent + "    "
        
        if hasattr(node, 'leftchd') and hasattr(node, 'rightchd'):
            print(f"{indent}├── True → Node {node.leftchd}")
            print_children(node.leftchd, next_indent)
            print(f"{indent}└── False → Node {node.rightchd}")
            print_children(node.rightchd, last_indent)
        elif hasattr(node, 'leftchd'):
            print(f"{indent}└── True → Node {node.leftchd}")
            print_children(node.leftchd, last_indent)
        elif hasattr(node, 'rightchd'):
            print(f"{indent}└── False → Node {node.rightchd}")
            print_children(node.rightchd, last_indent)
        else:
            # Leaf node
            print(f"{indent}└── Leaf (predcls: {node.predcls})")
    
    print_children(0)

# Run the simple tree visualization
simple_tree_viz(model)

In [ ]:
# Examine the data distribution in nodes 0, 1, and 2

def analyze_node_data_distribution(node_id):
    node = model[node_id]
    print(f"Node {node_id} Analysis:")
    print(f"  Predicted Class (predcls): {getattr(node, 'predcls', 'N/A')}")
    print(f"  Best Model Class (bstmdlclass): {getattr(node, 'bstmdlclass', 'N/A')}")
    print(f"  Gini Index (ginis): {getattr(node, 'ginis', 'N/A')}")
    print(f"  Validation Gini Index (ginist): {getattr(node, 'ginist', 'N/A')}")
    
    # Check class distribution in the node
    if hasattr(node, 'ycount'):
        print(f"  Training Data Class Distribution (ycount): {node.ycount}")
    if hasattr(node, attr):  # using the variable attr defined in the notebook ('ycountt')
        print(f"  Validation Data Class Distribution ({attr}): {getattr(node, attr)}")
    
    # Check indices of data points in this node
    print(f"  Training Indices: {len(getattr(node, 'trainidx', []))} points")
    print(f"  Test/Validation Indices: {len(getattr(node, 'testidx', []))} points")
    
    # For nodes with children, check split points
    if hasattr(node, 'trueidx') and hasattr(node, 'falseidx'):
        print(f"  True branch (trueidx): {len(node.trueidx)} training points")
        print(f"  False branch (falseidx): {len(node.falseidx)} training points")
    if hasattr(node, 'trueidxt') and hasattr(node, 'falseidxt'):
        print(f"  True branch validation (trueidxt): {len(node.trueidxt)} validation points")
        print(f"  False branch validation (falseidxt): {len(node.falseidxt)} validation points")
    
    print("\n")

print("Analyzing Node Data Distributions:\n")
analyze_node_data_distribution(0)
analyze_node_data_distribution(1)
analyze_node_data_distribution(2)

print("==== Detailed Class Distributions ====\n")

# Check if the sum of child node class distributions matches parent node
if hasattr(model[0], 'ycount') and hasattr(model[1], 'ycount') and hasattr(model[2], 'ycount'):
    total_children = model[1].ycount + model[2].ycount
    print("Checking consistency between Node 0 and its children (Nodes 1 & 2):")
    print(f"  Node 0 class distribution: {model[0].ycount}")
    print(f"  Sum of Nodes 1 & 2 distributions: {total_children}")
    print(f"  Match: {np.array_equal(model[0].ycount, total_children)}")

# Get predcls logic from train_utils.py
print("\nPredcls Assignment Logic:")
print("  Node predcls is determined from yoricountt.argmax() - the most common class in validation data")
print("  This is assigned before the Gini index is calculated or any split occurs")
print("  In other words, the predcls is assigned based on the node's validation data distribution")

# Check True/False split data details for Node 0
print("\nSplit Details for Node 0:")
if hasattr(model[0], 'trueidx') and hasattr(model[0], 'falseidx'):
    # Assuming Node 1 corresponds to the true branch and Node 2 to the false branch
    true_branch_points = len(getattr(model[1], 'trueidx', []))
    false_branch_points = len(getattr(model[2], 'falseidx', []))
    total_points = len(getattr(model[0], 'trainidx', []))
    print(f"  Training data in true branch (Node 1): {true_branch_points} points")
    print(f"  Training data in false branch (Node 2): {false_branch_points} points")
    print(f"  Total training data in Node 0: {total_points} points")
    if true_branch_points + false_branch_points != total_points:
        print("  ⚠️ INCONSISTENCY: Sum of branch data does not match total data")
    
    # Check if any branch is empty
    if len(getattr(model[0], 'trueidx', [])) == 0 or len(getattr(model[0], 'falseidx', [])) == 0:
        print("  ⚠️ One branch has no data points, which is consistent with a perfect split (Gini=0)")
    else:
        print("  Both branches have data points")


In [ ]:
# Inspect the actual data points that go into true and false branches
print("Class Distribution Analysis for Node 0 Split:")

# Get the actual classes of points in true and false branches
def check_class_distribution_in_branches(node_id):
    node = model[node_id]
    
    # Check training data
    if hasattr(node, 'trainidx') and hasattr(node, 'trueidx') and hasattr(node, 'falseidx'):
        print(f"\nNode {node_id} Training Data:")
        train_idx = node.trainidx
        true_idx = node.trueidx
        false_idx = node.falseidx
        
        print(f"  All points indices: {train_idx}")
        print(f"  True branch indices: {true_idx}")
        print(f"  False branch indices: {false_idx}")
        
        if hasattr(node, 'bstmdlclass'):
            print(f"\n  Best model class (bstmdlclass): {node.bstmdlclass}")
            
            # This checks the explanation for binary encoding in Ecdlabel function
            print(f"  Note: bstmdlclass = {node.bstmdlclass} means this model is trained to separate")
            print(f"        class {node.bstmdlclass} (value 1) from all other classes (value 0)")
        
    # Check validation data
    if hasattr(node, 'testidx') and hasattr(node, 'trueidxt') and hasattr(node, 'falseidxt'):
        print(f"\nNode {node_id} Validation Data:")
        test_idx = node.testidx
        true_idx = node.trueidxt
        false_idx = node.falseidxt
        
        print(f"  All points indices: {test_idx}")
        print(f"  True branch indices: {true_idx}")
        print(f"  False branch indices: {false_idx}")

check_class_distribution_in_branches(0)

# Let's investigate further based on what we've learned
# print("\nExplanation for Different Predicted Classes:")
# print("1. In Node 2, we have 11 data points: 10 of class 1 and 1 of class 2")
# print("2. Node 2's best model (TL_NN1) is trained to separate class 2 from others")
# print("3. The model successfully separates the single class 2 point (into Node 3) from all the class 1 points (into Node 4)")
# print("4. This perfect separation gives a training Gini index of 0 in Node 2")
# print("5. Node 3 contains only class 2 data, so its predcls=2")
# print("6. Node 4 contains only class 1 data, so its predcls=1")
# print("\nLooking at the code in Trainnode() function in train_utils.py:")
# print("predcls is assigned BEFORE model training by: Nodes[pronum].predcls = yoricountt.argmax()")
# print("bstmdlclass is assigned DURING model training when finding best model")
# print("\nConclusion:")
# print("The different predcls values in Nodes 3 and 4 reflect the perfect class separation achieved by Node 2's model.")
# print("Node 2's Gini=0 means that its best model (separating class 2 from others) perfectly")
# print("split the data points into pure class distributions in each child node.")


## Why Node 2 with Gini=0 has Children with Different Predicted Classes

We've uncovered the explanation for this apparent contradiction. The Gini index of 0 at Node 2 and the different predicted classes at Nodes 3 and 4 are actually consistent with how the NSTSC algorithm works:

### Key Insights:

1. **Node 2 Data Composition**: Node 2 contains 11 data points: 10 of class 1 and 1 of class 2

2. **Binary Classification at Each Node**: The NSTSC model trains a binary classifier at each node. In Node 2, the best model is a TL_NN1 (Conjunction/AND) model trained specifically to separate class 2 from all others.

3. **Perfect Separation Achieved**: This model perfectly separates the single class 2 instance (sending it to Node 3) from all the class 1 instances (sending them to Node 4). This is why the Gini index is 0 - the model has achieved a perfect separation.

4. **Different Predicted Classes**: 
   - Node 3 contains only class 2 data, so `predcls=2`
   - Node 4 contains only class 1 data, so `predcls=1`

### How the Gini Index Works in NSTSC:

The Gini index of 0 in Node 2 doesn't mean that all data points belong to the same class. Instead, it means:

- The **splitting rule** (the neural network model) perfectly separates the classes
- After the split, each child node contains data from only one class
- The weighted average impurity of the child nodes is 0

The `bstmdlclass` value of 2 for Node 2 tells us which class the model is trying to separate from the rest. It means "this model separates class 2 (positive) from all other classes (negative)." When this separation is perfect (as in this case), the Gini index becomes 0.

### Conclusion:

The different `predcls` values in Nodes 3 and 4 actually confirm that Node 2's split was perfect, which is consistent with its Gini index of 0. Rather than being a contradiction, this is exactly what we would expect from a perfect binary classifier that has successfully separated the classes.

## Understanding NSTSC Node Attributes

### Class Distribution Attributes

**`ycount` and `ycountt`:**
- **`ycount`**: Count of training data samples in each class for the current node
- **`ycountt`**: Count of validation/test data samples in each class for the current node

These arrays show how many samples of each class are present in the node. For example, if we have 3 classes (0, 1, 2), then `ycount = [5, 10, 2]` means there are 5 samples of class 0, 10 samples of class 1, and 2 samples of class 2 in this node's training data.

**`yoricount` and `yoricountt`:**
- **`yoricount`**: Original count of training samples per class before processing in the current node
- **`yoricountt`**: Original count of validation/test samples per class before processing in the current node

These are temporary variables used during node training in the `Trainnode()` function and are not stored as node attributes. They represent the class distribution before any splitting occurs.

### Classification Attributes

**`predcls` vs `bstmdlclass`:**

**`predcls` (Predicted Class):**
- The majority class in the node's validation data
- Determined by: `node.predcls = yoricountt.argmax()`
- Used as the final prediction if this becomes a leaf node
- Assigned BEFORE any model training occurs
- Based purely on class frequency, not model performance

**`bstmdlclass` (Best Model Class):**
- The class that the node's neural network model is trained to separate from others
- Only exists for internal nodes with a trained model
- Determined during model training when finding the split with minimum Gini index
- Represents which class the binary classifier is focused on (the "positive" class)
- When `bstmdlclass = 2`, the model separates class 2 (positive) from all others (negative)

### Important Distinction:

1. **`predcls`** is what the node would predict if it became a leaf node (based on majority class)
2. **`bstmdlclass`** is what the node's splitting model is trained to identify (based on best Gini improvement)

In a perfect split, the children nodes will have different `predcls` values that reflect the classes being separated by the parent node's model (which uses `bstmdlclass` to determine the split).

In [ ]:
# Demonstrating ycount, ycountt, predcls, and bstmdlclass with examples from our tree

# Print header and formatting helper
def format_node_info(node_id, show_detail=False):
    node = model[node_id]
    has_children = hasattr(node, 'leftchd') or hasattr(node, 'rightchd')
    node_type = "Internal" if has_children else "Leaf"
    
    print(f"\n{'=' * 50}")
    print(f"Node {node_id} ({node_type} Node) Details:")
    print(f"{'-' * 30}")
    
    # Show class distribution
    if hasattr(node, 'ycount'):
        print(f"ycount (training class distribution): {node.ycount}")
    if hasattr(node, 'ycountt'):
        print(f"ycountt (validation class distribution): {node.ycountt}")
        
    # Show predicted class and best model class
    if hasattr(node, 'predcls'):
        print(f"\npredcls (majority class for prediction): {node.predcls}")
        # Verify that predcls is the argmax of ycountt
        if hasattr(node, 'ycountt'):
            argmax_class = np.argmax(node.ycountt)
            print(f"Verification: argmax of ycountt = {argmax_class} {'✓' if argmax_class == node.predcls else '✗'}")
    
    if hasattr(node, 'bstmdlclass'):
        print(f"\nbstmdlclass (class to separate): {node.bstmdlclass}")
        print(f"This model separates class {node.bstmdlclass} from all others")
    
    # Show model type if available
    if hasattr(node, 'bestmodel'):
        from Models_node import TL_NN1, TL_NN2, TL_NN3, TL_NN4
        if isinstance(node.bestmodel, TL_NN1):
            model_type = "TL_NN1 (Conjunction/AND)"
        elif isinstance(node.bestmodel, TL_NN2):
            model_type = "TL_NN2 (Disjunction/OR)"
        elif isinstance(node.bestmodel, TL_NN3):
            model_type = "TL_NN3 (Always/Globally)"
        elif isinstance(node.bestmodel, TL_NN4):
            model_type = "TL_NN4 (Eventually/Finally)"
        else:
            model_type = "Unknown Model"
        print(f"Best Model: {model_type}")
        
    # Show children if available
    if hasattr(node, 'leftchd'):
        print(f"\nLeft Child: Node {node.leftchd} (True branch)")
        if hasattr(model[node.leftchd], 'predcls'):
            print(f"  - predcls: {model[node.leftchd].predcls}")
    if hasattr(node, 'rightchd'):
        print(f"Right Child: Node {node.rightchd} (False branch)")
        if hasattr(model[node.rightchd], 'predcls'):
            print(f"  - predcls: {model[node.rightchd].predcls}")
    
    # Show more detailed data if requested
    if show_detail:
        print("\nDetailed Data:")
        if hasattr(node, 'trainidx'):
            print(f"Training indices: {node.trainidx[:5]}{'...' if len(node.trainidx) > 5 else ''}")
        if hasattr(node, 'trueidx') and hasattr(node, 'falseidx'):
            print(f"True branch indices: {node.trueidx[:5]}{'...' if len(node.trueidx) > 5 else ''}")
            print(f"False branch indices: {node.falseidx[:5]}{'...' if len(node.falseidx) > 5 else ''}")

# Process all nodes in our model
for node_id in sorted(model.keys()):
    format_node_info(node_id)

# Show a diagram explaining predcls vs bstmdlclass
print("\n\n" + "=" * 80)
print("Conceptual Difference between predcls and bstmdlclass:")
print("-" * 80)
print("predcls: WHAT the node predicts if it's a leaf (majority class in validation data)")
print("bstmdlclass: HOW the node splits data (which class it separates from others)")
print("=" * 80)